# COLETA DEPUTADO

In [ ]:
import requests
import time
import csv
import pandas as pd
from collections import Counter

def coleta_lista_deputados():
    url = "https://dadosabertos.camara.leg.br/api/v2/deputados"
    resposta = requests.get(url)
    if resposta.status_code == 200:
        return resposta.json()['dados']
    return []

def coleta_deputado_info(id_deputado):
    url = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}"
    resposta_id_deputado = requests.get(url)
    if resposta_id_deputado.status_code == 200:
        dados = resposta_id_deputado.json()['dados']
        return {
            'nome': dados['nomeCivil'],
            'uf': dados['ultimoStatus']['siglaUf'],
            'partido': dados['ultimoStatus']['siglaPartido'],
            'email': dados['ultimoStatus']['gabinete']['email'],
        }
    return None

def coleta_lideres(uri_partido):
    if not uri_partido:
        return "Sem Líder", "Sem Presidente"

    id_partido = uri_partido.split('/')[-1]
    url = f"https://dadosabertos.camara.leg.br/api/v2/partidos/{id_partido}/lideres"
    resposta_lider = requests.get(url)
    lider, presidente = "Sem Lider", "Sem Presidente"
    if resposta_lider.status_code == 200:
        for p in resposta_lider.json()['dados']:
            if p['titulo'] == 'Líder': lider = p['nome']
            if p['titulo'] == 'Presidente': presidente = p['nome']
    return lider, presidente

def coleta_foco_politico(id_deputado):
    url_eventos = f"https://dadosabertos.camara.leg.br/api/v2/deputados/{id_deputado}/eventos?itens=15&ordem=DESC&ordenarPor=dataHoraInicio"
    resposta_eventos = requests.get(url_eventos)

    temas_acumulados = []
    total_eventos = 0
    eventos_com_temas = 0

    if resposta_eventos.status_code == 200:
        lista_eventos = resposta_eventos.json().get('dados', [])
        total_eventos = len(lista_eventos)

        for evento in lista_eventos:
            teve_tema_neste_evento = False
            id_evento = evento['id']
            url_pauta = f"https://dadosabertos.camara.leg.br/api/v2/eventos/{id_evento}/pauta"
            resposta_pauta = requests.get(url_pauta)

            if resposta_pauta.status_code == 200:
                itens_pauta = resposta_pauta.json().get('dados', [])
                for item in itens_pauta:
                    if item.get('proposicao_') and item['proposicao_'].get('id'):
                        id_prop = item['proposicao_']['id']
                        url_temas = f"https://dadosabertos.camara.leg.br/api/v2/proposicoes/{id_prop}/temas"
                        resposta_temas = requests.get(url_temas)

                        if resposta_temas.status_code == 200:
                            dados_temas = resposta_temas.json().get('dados', [])
                            if dados_temas:
                                teve_tema_neste_evento = True
                                for t in dados_temas:
                                    temas_acumulados.append(t['tema'])

            if teve_tema_neste_evento:
                eventos_com_temas += 1

    cobertura = f"[{eventos_com_temas}/{total_eventos} eventos com pauta]"

    if temas_acumulados:
        contagem = Counter(temas_acumulados)
        ranking = [f"{tema} ({freq})" for tema, freq in contagem.most_common()]
        return f"{cobertura} | " + " | ".join(ranking)

    return f"{cobertura} | Sem temas registrados"


lista = coleta_lista_deputados()

if lista:
    print("Gerando dados de perfil e eventos da API...")
    with open("analise_perfil_eventos.csv", "w", encoding='utf-8', newline='') as arquivo_csv:


        colunas = ['deputado_id', 'Nome', 'UF', 'Partido', 'Email', 'Líder', 'Presidente', 'Top Temas (Eventos/Freq)']

        escritor = csv.DictWriter(arquivo_csv, fieldnames=colunas)
        escritor.writeheader()

        # mude para aumentar o range e pegar mais deputados
        for deputado in lista[:5]:
            id_atual = deputado['id']
            nome_dep = deputado['nome']
            print(f"Analisando: {nome_dep} (ID: {id_atual})...")
            time.sleep(0.1)
            try:
                perfil = coleta_deputado_info(id_atual)
                if not perfil: continue

                nome_lider, nome_pres = coleta_lideres(deputado['uriPartido'])
                ranking_temas_eventos = coleta_foco_politico(id_atual)

                escritor.writerow({
                    "deputado_id": id_atual,
                    "Nome": perfil['nome'],
                    "UF": perfil['uf'],
                    "Partido": perfil['partido'],
                    "Email": perfil['email'],
                    "Líder": nome_lider,
                    "Presidente": nome_pres,
                    "Top Temas (Eventos/Freq)": ranking_temas_eventos
                })
            except Exception as e:
                print(f"Erro ao processar {nome_dep}: {e}")

    print(" Dados da API coletados.")

    # =========================================================
    # Junta arquivos
    # =========================================================
    print("\n Iniciando a fusão ")
    try:

        df_perfil = pd.read_csv("analise_perfil_eventos.csv")
        df_presenca = pd.read_csv("modulo2_presenca_completa.csv")
        df_proposicoes = pd.read_csv("modulo1_proposicoes_completa.csv")
        df_despesas = pd.read_csv("modulo3_despesas_completa.csv")
        df_impacto = pd.read_csv("modulo1_5_impacto_pronto.csv")

        df_presenca = df_presenca.drop(columns=['deputado_nome'], errors='ignore')
        df_proposicoes = df_proposicoes.drop(columns=['deputado_nome'], errors='ignore')


        df_final = pd.merge(df_perfil, df_presenca, on='deputado_id', how='left')
        df_final = pd.merge(df_final, df_proposicoes, on='deputado_id', how='left')
        df_final = pd.merge(df_final, df_despesas, on='deputado_id', how='left')
        df_final = pd.merge(df_final, df_impacto, on='deputado_id', how='left')

        df_final = df_final.fillna(0)


        df_final.to_csv("ARQUIVAO_FINAL_COLAB.csv", index=False, encoding='utf-8')
        print(" ARQUIVÃO FINAL SUCESSO! ('ARQUIVAO_FINAL_COLAB.csv')")

    except FileNotFoundError as e:
        print(f"\n Faltou um arquivo para fazer a fusão: {e}")
        print("Certifique-se de ter rodado os Módulos 1, 2 e 3 antes de executar o principal!")